# Experimento: classificar suporte ao contexto com TF-IDF

Este notebook testa uma tarefa **diferente da classificação de emoções**: prever se a resposta esperada está apoiada pelo contexto fornecido (`positivo` ou `negativo`). Usa `data/datasets/faq_qa/` e compara uma MLP com TF-IDF a uma regressão logística.

A entrada do modelo contém somente o contexto e a pergunta. A mensagem `assistant` e outros metadados não entram nas features. O vocabulário e a seleção de features são ajustados em `train`; a validação escolhe o limiar; `test` fica reservado para a avaliação final. Este experimento não integra nem altera o tutor.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.neural_network import MLPClassifier

SEED = 42
N_FEATURES = 5000

def localizar_raiz() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data" / "datasets" / "faq_qa" / "train.jsonl").is_file():
            return candidate
    raise FileNotFoundError(
        "Abra o notebook a partir da raiz do repositório calma-ai."
    )

ROOT = localizar_raiz()
DATA_DIR = ROOT / "data" / "datasets" / "faq_qa"
print(f"Raiz do projeto: {ROOT}")
print(f"Dataset: {DATA_DIR}")

Raiz do projeto: c:\Users\maximus.nascimento\Downloads\calma-ai
Dataset: c:\Users\maximus.nascimento\Downloads\calma-ai\data\datasets\faq_qa


## Carregar os splits

Falhas de formato e rótulos inesperados geram erro explícito. Os IDs são conferidos para garantir que os splits não se sobreponham.

In [6]:
CONTEXTO_PERGUNTA = re.compile(
    r"(?ims)^\s*Contexto:\s*(.*?)^\s*Pergunta:\s*(.+?)\s*$"
)
ROTULOS = {"negativo": 0, "positivo": 1}

def carregar_split(nome: str) -> dict:
    caminho = DATA_DIR / f"{nome}.jsonl"
    textos, alvos, ids = [], [], []
    for numero_linha, linha in enumerate(caminho.read_text(encoding="utf-8").splitlines(), 1):
        if not linha.strip():
            continue
        try:
            registro = json.loads(linha)
            mensagens = registro["messages"]
            usuario = next(m["content"] for m in mensagens if m["role"] == "user")
            metadados = registro["meta"]
            rotulo = metadados["tipo"]
            identificador = metadados["id"]
        except (KeyError, StopIteration, TypeError, json.JSONDecodeError) as erro:
            raise ValueError(f"Registro inválido em {caminho}:{numero_linha}") from erro

        match = CONTEXTO_PERGUNTA.search(usuario)
        if not match:
            raise ValueError(f"Contexto/pergunta ausente em {caminho}:{numero_linha}")
        if rotulo not in ROTULOS:
            raise ValueError(f"Rótulo inesperado em {caminho}:{numero_linha}: {rotulo!r}")
        if not isinstance(identificador, str) or not identificador:
            raise ValueError(f"ID ausente em {caminho}:{numero_linha}")

        contexto, pergunta = match.groups()
        textos.append(f"Contexto: {contexto.strip()}\nPergunta: {pergunta.strip()}")
        alvos.append(ROTULOS[rotulo])
        ids.append(identificador)

    if not textos:
        raise ValueError(f"Split vazio: {caminho}")
    return {"textos": textos, "y": np.asarray(alvos, dtype=np.int64), "ids": ids}

dados = {nome: carregar_split(nome) for nome in ("train", "val", "test")}
for nome in ("train", "val"):
    print(f"{nome}: {len(dados[nome]['y'])} exemplos; rótulos 0/1 = "
          f"{np.bincount(dados[nome]['y'], minlength=2).tolist()}")
print(f"test: {len(dados['test']['y'])} exemplos; reservado para avaliação final.")

for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    sobrepostos = set(dados[a]["ids"]) & set(dados[b]["ids"])
    if sobrepostos:
        raise ValueError(f"IDs compartilhados entre {a} e {b}: {sorted(sobrepostos)[:5]}")
print("IDs dos três splits não se sobrepõem.")

train: 640 exemplos; rótulos 0/1 = [160, 480]
val: 136 exemplos; rótulos 0/1 = [34, 102]
test: 132 exemplos; reservado para avaliação final.
IDs dos três splits não se sobrepõem.


## TF-IDF e seleção de features

O vocabulário e o teste qui-quadrado são ajustados apenas com `train`. O split de teste ainda não é transformado nem consultado.

In [7]:
word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), min_df=2, max_features=30000, sublinear_tf=True
)
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(2, 5), min_df=3,
    max_features=30000, sublinear_tf=True
)

X_train_full = hstack([
    word_vectorizer.fit_transform(dados["train"]["textos"]),
    char_vectorizer.fit_transform(dados["train"]["textos"]),
]).tocsr()
X_val_full = hstack([
    word_vectorizer.transform(dados["val"]["textos"]),
    char_vectorizer.transform(dados["val"]["textos"]),
]).tocsr()

seletor = SelectKBest(chi2, k=min(N_FEATURES, X_train_full.shape[1]))
X_train = seletor.fit_transform(X_train_full, dados["train"]["y"])
X_val = seletor.transform(X_val_full)
print(f"Features: {X_train_full.shape[1]:,} -> {X_train.shape[1]:,}")

Features: 23,747 -> 5,000


## Treino e calibração na validação

A MLP é uma rede neural de uma camada oculta. A regressão logística fornece uma referência simples. Os limiares são escolhidos por F1 somente em `val`.

In [8]:
modelos = {
    "regressao_logistica": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=SEED
    ),
    "mlp_tfidf": MLPClassifier(
        hidden_layer_sizes=(64,), activation="relu", solver="adam",
        alpha=1e-4, batch_size=64, learning_rate_init=1e-3,
        max_iter=150, early_stopping=True, validation_fraction=0.15,
        n_iter_no_change=12, random_state=SEED,
    ),
}

def limiar_por_f1_macro(y_true: np.ndarray, probabilidades: np.ndarray) -> float:
    candidatos = np.unique(np.concatenate(([0.0, 0.5, 1.0], probabilidades)))
    melhor = None
    for limiar in candidatos:
        pred = (probabilidades >= limiar).astype(np.int64)
        chave = (
            f1_score(y_true, pred, average="macro", zero_division=0),
            balanced_accuracy_score(y_true, pred),
            float(limiar),
        )
        if melhor is None or chave > melhor[0]:
            melhor = (chave, float(limiar))
    if melhor is None:
        raise ValueError("Não foi possível calibrar o limiar na validação.")
    return melhor[1]

resultados = {}
for nome, modelo in modelos.items():
    modelo.fit(X_train, dados["train"]["y"])
    prob_val = modelo.predict_proba(X_val)[:, 1]
    limiar = limiar_por_f1_macro(dados["val"]["y"], prob_val)
    pred_val = (prob_val >= limiar).astype(np.int64)
    resultados[nome] = {"modelo": modelo, "limiar": limiar}
    print(f"{nome}: limiar={limiar:.3f}; F1 macro val="
          f"{f1_score(dados['val']['y'], pred_val, average='macro'):.3f}")

regressao_logistica: limiar=0.568; F1 macro val=0.754
mlp_tfidf: limiar=0.820; F1 macro val=0.690


## Avaliação final em teste

Execute esta célula somente depois de concluir as decisões de modelo e limiar com a validação. As métricas abaixo usam `test`, que não participou do treino nem da calibração.

In [9]:
X_test_full = hstack([
    word_vectorizer.transform(dados["test"]["textos"]),
    char_vectorizer.transform(dados["test"]["textos"]),
]).tocsr()
X_test = seletor.transform(X_test_full)
y_test = dados["test"]["y"]

for nome, resultado in resultados.items():
    probabilidades = resultado["modelo"].predict_proba(X_test)[:, 1]
    pred = (probabilidades >= resultado["limiar"]).astype(np.int64)
    metricas = {
        "limiar (val)": resultado["limiar"],
        "F1 macro": f1_score(y_test, pred, average="macro"),
        "F1 positivo": f1_score(y_test, pred, pos_label=1, zero_division=0),
        "precisao positivo": precision_score(y_test, pred, pos_label=1, zero_division=0),
        "recall positivo": recall_score(y_test, pred, pos_label=1, zero_division=0),
        "balanced accuracy": balanced_accuracy_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, probabilidades),
        "PR-AUC": average_precision_score(y_test, probabilidades),
    }
    print(f"\n{nome}")
    for chave, valor in metricas.items():
        print(f"  {chave}: {valor:.3f}")
    print("  matriz de confusao [linhas=real, colunas=previsto]:")
    print(confusion_matrix(y_test, pred, labels=[0, 1]))
    print(classification_report(
        y_test, pred, labels=[0, 1], target_names=["negativo", "positivo"],
        zero_division=0,
    ))


regressao_logistica
  limiar (val): 0.568
  F1 macro: 0.733
  F1 positivo: 0.800
  precisao positivo: 1.000
  recall positivo: 0.667
  balanced accuracy: 0.833
  ROC-AUC: 0.792
  PR-AUC: 0.939
  matriz de confusao [linhas=real, colunas=previsto]:
[[33  0]
 [33 66]]
              precision    recall  f1-score   support

    negativo       0.50      1.00      0.67        33
    positivo       1.00      0.67      0.80        99

    accuracy                           0.75       132
   macro avg       0.75      0.83      0.73       132
weighted avg       0.88      0.75      0.77       132


mlp_tfidf
  limiar (val): 0.820
  F1 macro: 0.693
  F1 positivo: 0.795
  precisao positivo: 0.909
  recall positivo: 0.707
  balanced accuracy: 0.747
  ROC-AUC: 0.763
  PR-AUC: 0.927
  matriz de confusao [linhas=real, colunas=previsto]:
[[26  7]
 [29 70]]
              precision    recall  f1-score   support

    negativo       0.47      0.79      0.59        33
    positivo       0.91      0.71      0